# Tech Challenge 3 - Big Data & Analytics

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import os
import matplotlib.pyplot as plt

In [17]:
base_path = "../data"
arquivos = [
    "dataset-2025-2026.csv",
    "dataset-2024-2025.csv",
    "dataset-2023-2024.csv",
]

for arquivo in arquivos:
    caminho = os.path.join(base_path, arquivo)
    df = pd.read_csv(caminho, low_memory=False, sep=",", encoding="latin-1", on_bad_lines="skip")
    print(f"\n=== {arquivo} ===")
    print(f"Linhas: {len(df)} | Colunas: {len(df.columns)}")
    print(f"Colunas: {list(df.columns)}")


=== dataset-2025-2026.csv ===
Linhas: 3495 | Colunas: 388
Colunas: ['0.a_token', '0.d_data/hora_envio', '1.a_idade', '1.a.1_faixa_idade', '1.b_genero', '1.c_cor/raca/etnia', '1.d_pcd', '1.e_experiencia_profissional_prejudicada', '1.e.1_NÃ£o acredito que minha experiÃªncia profissional seja afetada', '1.e.2_Sim, devido a minha Cor/RaÃ§a/Etnia', '1.e.3_Sim, devido a minha identidade de gÃªnero', '1.e.4_Sim, devido ao fato de ser PCD', '1.f_aspectos_prejudicados', '1.f.1_Quantidade de oportunidades de emprego/vagas recebidas', '1.f.2_Senioridade das vagas recebidas em relaÃ§Ã£o Ã\xa0 sua experiÃªncia', '1.f.3_AprovaÃ§Ã£o em processos seletivos/entrevistas', '1.f.4_Oportunidades de progressÃ£o de carreira', '1.f.5_Velocidade de progressÃ£o de carreira', '1.f.6_NÃ\xadvel de cobranÃ§a no trabalho/Stress no trabalho', '1.f.7_AtenÃ§Ã£o dada pelas pessoas diante das minhas opiniÃµes e ideias', '1.f.8_RelaÃ§Ã£o com outras pessoas da empresa, em momentos de trabalho', '1.f.9_RelaÃ§Ã£o com outras

## Perguntas que precisam ser respondidas
- Como está estruturado o mercado brasileiro de Dados?
- Quais perfis profissionais são mais valorizados pelo mercado?
- Qual é o cenário de diversidade de gênero nas carreiras de dados?
- Quais tecnologias apresentam maior adoção entre os profissionais?
- Qual é o índice de adoção de Inteligência Artificial e seu impacto?
- Existem diferenças relevantes entre regiões, senioridades ou modelos de
trabalho?
- Quais oportunidades e desafios podem ser identificados para empresas
que desejam investir em Dados e Inteligência Artificial?

## Passo a Passo - Data Analytics Pipeline

### Pré-requisitos

- Conta no AWS Academy Lab (fornecida pela pós)
- Python 3.9+ instalado localmente
- AWS CLI instalado
- Bibliotecas: `boto3`, `pandas`, `matplotlib`, `seaborn`

### Configuração AWS

```bash
brew install aws

aws configure
```
Os dados pedidos no momento do `aws configure` estão disponíveis no lab:

Cursos -> Módulos -> AWS Details -> AWS CLI -> Show

### Cria o S3 e cria as pastas

```bash
BUCKET="bucket-tech-challenge-g10"

# Criar o bucket
aws s3 mb s3://$BUCKET --region us-east-1

# Criar as "pastas" (prefixos) para cada camada
aws s3api put-object --bucket $BUCKET --key raw/state-of-data/
aws s3api put-object --bucket $BUCKET --key cleansed/state-of-data/
aws s3api put-object --bucket $BUCKET --key transformed/state-of-data/
aws s3api put-object --bucket $BUCKET --key curated/state-of-data/
aws s3api put-object --bucket $BUCKET --key archives/scripts/
```
### Copia os dados brutos locais para nuvem

```bash

# Subir os 3 arquivos brutos para a Raw
aws s3 cp "<caminho-arquivo-local>" "s3://$BUCKET/raw/state-of-data/"
aws s3 cp "<caminho-arquivo-local>" "s3://$BUCKET/raw/state-of-data/"
aws s3 cp "<caminho-arquivo-local>" "s3://$BUCKET/raw/state-of-data/"

# Verificar se subiu corretamente
aws s3 ls s3://$BUCKET/raw/ --recursive
```

### Subir o arquivo de script no Glue 

1. Na console, busque por AWS Glue
2. Vá em ETL Jobs
3. Escolha a opção "Script editor"
4. Cole o script na aba dedicada a isso 
5. Configure os detalhes no "Job details", como o s3 onde salvar o script
6. Após configurar e salvar, rode com o comando:
```bash
aws glue start-job-run \
  --job-name <nome-do-job> \
  --arguments '{"--BUCKET":"<nome-do-bucket>"}' \
  --region <regiao>
```
7. Acompanhe o job na console em "Runs"

### Configure e execute o Crawler

Após rodar o job com sucesso, é necessário configurar e executar o Crawler para que o AWS Glue Catalog catalogue a estrutura da tabela. Ela será necessária para futura adoção do Athena.

### Passo a passo de como criar um Crawler

Para criar Crawler, vá na console da AWS e pesquise por "AWS Glue"
1. No menu lateral, na seção "Data Catalog" selecione "Crawlers"
2. Selecione "Create Crawler"
3. Informe o nome desejado e clique em "Next"
4. Em "Data source configuration" selecione a opção "Not Yet"
5. Na sequência clique em "Add a data source" e configure com os dados desejados. No exemplo de criação do curated seleciono a opção S3 e em "S3 path" seleciono a caminho da onde estará os dados do curated
6. Clique em "Add an S3 data source"
7. Next
8. Já em IAM Role usaremos a mesma criada automaticamente pela AWS Academy Lab que é a `LabRole`
9. Next
10. Já em "Output configuration" se é a primeira vez que criamos, temos que adicionar um novo database clicando em "Add database". Essa fase é simples e só informamos o nome do banco mantendo o type como `Glue Database`.
11. Next
12. Revisar e criar. 
13. Está pronto teu Crawler e é só executar! 

> 💡 **NOTA:**
> Ele é sob-demanda, então sempre que rodar um ETL job será necessário rodar o Crawler novamente.

## Criando e configurando AWS Athena

1. No Athena, abra Query editor.
2. Selecione o database criado previamente
3. Vá em Settings (ou Manage settings).
4. Em Query result location, informe: s3://bucket-tech-challenge-g10/athena-results/
5. Salve.
6. Rode as queries SQL.